<a href="https://colab.research.google.com/github/ivanadelgado/Yimi-Reynaga-/blob/main/preqca_yimy_reynaga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# preQCA — Preparar los datos para fsQCA

QCA necesita representar las condiciones y el resultado como **pertenencias a conjuntos**.

En este notebook trabajaremos en paralelo con la guía: conoceremos las mediciones originales, estableceremos los umbrales, calibraremos las mediciones, compararemos los valores originales con las membresías y finalmente guardaremos la base preparada.

**Unidad de análisis:** cada proceso de reforma constitucional.

`mediciones originales → conceptualización → umbrales → calibración → membresías`


## Instalar la librería

En Google Colab necesitamos instalar `setqca` una vez por sesión.


In [ ]:
!pip -q install setqca


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 1.0 MB/s eta 0:00:00


## preQCA-01: conocer las mediciones originales

Cada fila representa un **proceso de reforma constitucional**.

Las variables todavía son **mediciones empíricas**, no membresías QCA.

Antes de calibrarlas, la guía discute para cada variable su definición, importancia teórica, fuente, rango, referencia y temporalidad.


In [ ]:
import pandas as pd
from setqca import calibrate_direct

linkGit="https://github.com/ivanadelgado/Yimi-Reynaga-/raw/refs/heads/main/WVS7_confianza_parlamento_1.csv"
qca_raw = pd.read_csv(linkGit)

qca_raw.head()


,pais,confianza_parlamento,magnitud_distrito,electores_por_distrito,duracion_mandato
0,Albania,7.6,11.67,287694.0,4
1,Andorra,36.7,2.00,6820.0,4
2,Argentina,20.3,10.71,1425753.0,4
3,Australia,27.6,1.00,108739.0,3
4,Austria,43.6,1.84,391642.0,5


## De las mediciones a los conjuntos

A partir de las mediciones originales construiremos cuatro conjuntos:

- `LEGITIMIDAD`: agregar definición;
- `MAGNITUD`: agregar definición;
- `ELECTORES`: agregar definición;
- `DURACIÓN`: agregar definición.

La medición original y el conjunto no son lo mismo. La calibración permitirá traducir una medición empírica en un grado de pertenencia entre 0 y 1.


## preQCA-02: establecer los umbrales

En este ejercicio los umbrales son decisiones simuladas con fines pedagógicos.

| Conjunto | Fuera | Cruce | Dentro |
|---|---:|---:|---:|
| LEGITIMIDAD | 20 | 35 | 50 |
| MAGNITUD | 6 | - | - |
| ELECTORES | 500000 | 350000 | 200000 |
| DURACIÓN | 5 | 4 | 2 |

Estos valores definen cómo interpretaremos cada medición para construir su conjunto. No dependen de las configuraciones que QCA produzca después.


In [ ]:
umbrales = {
    "LEGITIMIDAD": (20, 35, 50),
    "MAGNITUD": (6, 4, 2),
    "ELECTORES": (500000, 350000, 200000),
    "DURACION": (5, 4, 2)
}

umbrales


{'LEGITIMIDAD': (20, 35, 50),
 'MAGNITUD': (6, 4, 2),
 'ELECTORES': (500000, 350000, 200000),
 'DURACION': (5, 4, 2)}

## preQCA-03: calibrar las mediciones

`calibrate_direct()` recibe una medición y tres referencias sustantivas:

- `full_out`: claramente fuera del conjunto;
- `crossover`: punto de máxima ambigüedad;
- `full_in`: claramente dentro del conjunto.

Aplicamos la calibración por separado porque cada conjunto tiene sus propios umbrales.


In [ ]:
qca = pd.DataFrame({
    "pais": qca_raw["pais"],

    "LEGITIMIDAD": calibrate_direct(
        qca_raw["confianza_parlamento"],
        full_out=-20,
        crossover=35,
        full_in=50
    ),

    "MAGNITUD": calibrate_direct(
        qca_raw["magnitud_distrito"],
        full_out=6,
        crossover=50,
        full_in=70
    ),

    "ELECTORES": calibrate_direct(
        qca_raw["electores_por_distrito"],
        full_out=500000,
        crossover=350000,
        full_in=200000
    ),

    "DURACION": calibrate_direct(
        qca_raw["duracion_mandato"],
        full_out=5,
        crossover=4,
        full_in=2
    )
})

qca.head()


,pais,LEGITIMIDAD,MAGNITUD,ELECTORES,DURACION
0,Albania,0.187419,0.071425,7.725983e-01,0.500000
1,Andorra,0.582660,0.038712,9.988146e-01,0.500000
2,Argentina,0.312820,0.067279,6.748073e-10,0.500000
3,Australia,0.402235,0.036297,9.913014e-01,0.813395
4,Austria,0.843980,0.038316,3.063125e-01,0.050000


## preQCA-04: comparar mediciones y membresías

Ahora observamos lado a lado las **mediciones originales** y las **membresías calibradas**.

`medición original → pertenencia al conjunto`


In [ ]:
comparacion = pd.DataFrame({
    "pais": qca_raw["pais"],
    "confianza_parlamento": qca_raw["confianza_parlamento"],
    "LEGITIMIDAD": qca["LEGITIMIDAD"],
    "magnitud_distrito": qca_raw["magnitud_distrito"],
    "MAGNITUD": qca["MAGNITUD"],
    "electores_por_distrito": qca_raw["electores_por_distrito"],
    "ELECTORES": qca["ELECTORES"],
    "duracion_mandato": qca_raw["duracion_mandato"],
    "DURACION": qca["DURACION"]
})

comparacion.head(10)

,pais,confianza_parlamento,LEGITIMIDAD,magnitud_distrito,MAGNITUD,electores_por_distrito,ELECTORES,duracion_mandato,DURACION
0,Albania,7.6,0.187419,11.67,0.071425,287694.0,7.725983e-01,4,0.500000
1,Andorra,36.7,0.582660,2.00,0.038712,6820.0,9.988146e-01,4,0.500000
2,Argentina,20.3,0.312820,10.71,0.067279,1425753.0,6.748073e-10,4,0.500000
3,Australia,27.6,0.402235,1.00,0.036297,108739.0,9.913014e-01,3,0.813395
4,Austria,43.6,0.843980,1.84,0.038316,391642.0,3.063125e-01,5,0.050000
5,Bangladesh,74.7,0.999588,1.00,0.036297,347141.0,5.140266e-01,5,0.050000
6,Armenia,22.5,0.338678,5.00,0.046915,369674.0,4.046342e-01,5,0.050000
7,Bolivia,9.2,0.200815,1.00,0.036297,179125.0,9.662423e-01,5,0.050000
8,Bosnia Herzegovina,16.2,0.267673,5.25,0.047669,671086.0,1.827846e-03,4,0.500000
9,Brazil,15.7,0.262458,19.00,0.111601,5435205.0,0.000000e+00,4,0.500000


## preQCA-05: guardar la base preparada

Una vez revisada la calibración, guardamos las membresías en un archivo separado.

Este archivo será la entrada del notebook `QCA`.


In [ ]:
qca.to_csv("qca_calibrada.csv", index=False)

print("Archivo creado: qca_calibrada.csv")
print("Casos:", len(qca))


Archivo creado: qca_calibrada.csv
Casos: 74
